# Sprint 3: SOTA Upgrade (Memory Efficient Pipeline)

Notebook ini mengimplementasikan perbaikan besar-besaran (Sprint 3) untuk menyamakan performa dengan project SOTA MSCNN-BiLSTM-AE.

## Key Improvements:
1. **Preprocessing Baru:** MinMax Scaler (0-1) yang bersih, fit hanya pada Benign Train.
2. **Clipping:** Menangani outlier ekstrim di data Test dengan clipping ke [0, 1].
3. **Memory Efficient:** Menggunakan `tf.data.Dataset` generator untuk streaming data (anti-OOM).
4. **Modular:** Menggunakan script terpisah untuk preprocessing yang konsisten.

In [ ]:
# @title Colab Bootstrap (with Drive Persistence)
# Jalankan cell ini jika di Google Colab untuk setup environment
from pathlib import Path
import os
import subprocess
import sys
import shutil

COLAB_BOOTSTRAP_ENABLE = True
COLAB_REPO_URL = "https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git"
COLAB_BRANCH = "feat/sprint3-sota-upgrade"
COLAB_REPO_DIR = Path("/content/nids-cnn-lstm-autoencoder")

# Drive Persistence Config
COLAB_DRIVE_MOUNT = Path("/content/drive")
COLAB_PROJECT_DRIVE_ROOT = COLAB_DRIVE_MOUNT / "MyDrive/nids-cnn-lstm-autoencoder"

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _run_shell(cmd: list[str], cwd: Path | None = None) -> None:
    print(f"[CMD] {' '.join(cmd)}")
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def _ensure_symlink_dir(repo_path: Path, drive_target: Path) -> None:
    """Symlink repo folder to drive folder for persistence."""
    drive_target.mkdir(parents=True, exist_ok=True)
    
    if repo_path.is_symlink():
        if repo_path.resolve() == drive_target.resolve():
            print(f"[INFO] Symlink OK: {repo_path} -> {drive_target}")
            return
        repo_path.unlink()
    elif repo_path.exists():
        if any(repo_path.iterdir()):
            print(f"[WARN] Folder exists and not empty: {repo_path}. Skipping link.")
            return
        repo_path.rmdir()
        
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(drive_target), str(repo_path), target_is_directory=True)
    print(f"[INFO] Symlink created: {repo_path} -> {drive_target}")

if _is_colab_runtime() and COLAB_BOOTSTRAP_ENABLE:
    from google.colab import drive
    drive.mount(str(COLAB_DRIVE_MOUNT))
    
    if not COLAB_REPO_DIR.exists():
        _run_shell(["git", "clone", "-b", COLAB_BRANCH, COLAB_REPO_URL, str(COLAB_REPO_DIR)])
    else:
        _run_shell(["git", "fetch", "--all"], cwd=COLAB_REPO_DIR)
        _run_shell(["git", "checkout", COLAB_BRANCH], cwd=COLAB_REPO_DIR)
        _run_shell(["git", "pull", "origin", COLAB_BRANCH], cwd=COLAB_REPO_DIR)
        
    # PERSISTENCE: Link Sprint 3 Data & Models to Drive
    # 1. Processed Data
    _ensure_symlink_dir(
        COLAB_REPO_DIR / "data/research/sprint3_upgrade/processed",
        COLAB_PROJECT_DRIVE_ROOT / "data/research/sprint3_upgrade/processed"
    )
    # 2. Models & Results
    _ensure_symlink_dir(
        COLAB_REPO_DIR / "research/sprint3_upgrade/models",
        COLAB_PROJECT_DRIVE_ROOT / "research/sprint3_upgrade/models"
    )
    _ensure_symlink_dir(
        COLAB_REPO_DIR / "research/sprint3_upgrade/results",
        COLAB_PROJECT_DRIVE_ROOT / "research/sprint3_upgrade/results"
    )
        
    # Set Working Directory
    os.chdir(COLAB_REPO_DIR)
    print(f"[INFO] Working Directory set to: {os.getcwd()}")
else:
    print("Not in Colab or Bootstrap disabled.")

In [ ]:
# Setup Project Root & Imports
import os
import sys
from pathlib import Path
import yaml
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import glob
from tqdm.notebook import tqdm

# Deteksi Project Root (Fallback mechanism)
try:
    # Jika di Colab dan sudah chdir, getcwd() adalah root
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    
    # Validasi sederhana: cek folder research ada atau tidak
    if not (PROJECT_ROOT / "research").exists():
        # Coba naik satu level (jika notebook dijalankan lokal dari folder notebooks/)
        if (PROJECT_ROOT.parent / "research").exists():
            PROJECT_ROOT = PROJECT_ROOT.parent
        else:
            # Fallback hardcoded untuk Colab jika chdir gagal
            PROJECT_ROOT = Path("/content/nids-cnn-lstm-autoencoder")
except Exception:
    PROJECT_ROOT = Path(".")

print(f"Project Root: {PROJECT_ROOT}")
sys.path.append(str(PROJECT_ROOT))

# Load Config Sprint 3
config_path = PROJECT_ROOT / "research/sprint3_upgrade/config/experiment_v1.yaml"

if config_path.exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print(f"Experiment: {config['experiment_name']}")
    print(f"Config Loaded: {config_path}")
else:
    print(f"ERROR: Config not found at {config_path}")
    print("Make sure you have cloned the repo and switched to the correct branch.")

## 1. Data Inspection (EDA)
Menganalisis distribusi data mentah (RAW) atau processed yang tersedia.
Cell ini akan mencoba mendeteksi data secara otomatis di path standar.

In [ ]:
# @title Data Inspection & Class Distribution
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm
import numpy as np
import os
import glob
from pathlib import Path

# Default values to prevent NameError
train_files = []
test_files = []

# --- AUTO DETECT DATA PATHS ---
# Coba deteksi path data processed (hasil run sebelumnya)
try:
    base_processed = None
    if 'config' in locals() and 'paths' in config:
        if 'processed_data' in config['paths']:
            base_processed = PROJECT_ROOT / config['paths']['processed_data']
        elif 'train_data' in config['paths']:
            base_processed = Path(config['paths']['train_data']).parent

    if base_processed is None:
        candidates = [
            Path("data/processed"),
            Path("../data/processed"),
            Path("/content/nids-cnn-lstm-autoencoder/data/research/sprint3_upgrade/processed"),
            Path("/content/nids-mscnn-bilstm-autoencoder/data/processed")
        ]
        for p in candidates:
            if p.exists():
                base_processed = p
                break

    if base_processed:
        print(f"[INFO] Inspecting data at: {base_processed}")
        train_files = sorted(glob.glob(str(base_processed / "train" / "*.npz")))
        test_files = sorted(glob.glob(str(base_processed / "test" / "*.npz")))
        
        if not train_files:
             train_files = sorted(glob.glob(str(base_processed / "**" / "train" / "*.npz"), recursive=True))
        if not test_files:
             test_files = sorted(glob.glob(str(base_processed / "**" / "test" / "*.npz"), recursive=True))
    else:
        print("[WARN] Processed data directory not found yet. Run preprocessing first or check paths.")
        
except Exception as e:
    print(f"[ERROR] Path detection failed: {e}")

def analyze_distribution(files, set_name="Dataset"):
    print(f"\n--- Analyzing {set_name} ---")
    if not files:
        print("No files found.")
        return

    try:
        with np.load(files[0], allow_pickle=True) as data:
            print(f"Sample File: {os.path.basename(files[0])}")
            print(f"Keys: {list(data.keys())}")
            for k in data.keys():
                obj = data[k]
                print(f"  {k}: shape={obj.shape}, dtype={obj.dtype}")
    except Exception as e:
        print(f"Error reading sample file: {e}")
        return

    label_counts = Counter()
    total_samples = 0
    
    scan_limit = min(len(files), 100) 
    print(f"Scanning {scan_limit} files (sample) for label distribution...")
    
    for f in tqdm(files[:scan_limit], desc=f"Scanning {set_name}"):
        try:
            with np.load(f, allow_pickle=True) as data:
                y = data['y'] if 'y' in data else (data['Y'] if 'Y' in data else None)
                if y is not None:
                    if len(y.shape) > 0:
                        label_counts.update(y)
                    else:
                        label_counts[y.item()] += 1
                    total_samples += len(y) if len(y.shape) > 0 else 1
        except: pass
        
    scale_factor = len(files) / scan_limit
    estimated_total = int(total_samples * scale_factor)
    
    print(f"Scanned Samples: {total_samples}")
    print(f"Estimated Total Samples: {estimated_total}")
    print(f"Label Distribution (Scanned): {dict(label_counts)}")
    
    if label_counts:
        plt.figure(figsize=(10, 5))
        keys = list(label_counts.keys())
        vals = list(label_counts.values())
        sns.barplot(x=keys, y=vals)
        plt.title(f"Label Distribution (Sampled) - {set_name}")
        plt.xlabel("Label (0=Benign, 1=Attack)")
        plt.ylabel("Count")
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.show()

if train_files:
    analyze_distribution(train_files, "Train Set (Benign Expected)")
else:
    print("Train files not found. (Maybe preprocessing hasn't run yet?)")

if test_files:
    analyze_distribution(test_files, "Test Set (Mixed Expected)")
else:
    print("Test files not found.")


## 1. Preprocessing (SOTA Style)
Jalankan script preprocessing baru yang menerapkan MinMax(0-1) dan Clipping.

In [ ]:
# Run Preprocessing Script (If data not exists)
processed_dir = PROJECT_ROOT / config['paths']['processed_data']
train_dir = processed_dir / "train"
test_dir = processed_dir / "test"

RUN_PREPROCESSING = False 

# --- AUTO DETECT & REUSE LOGIC ---
# Check if processed data already exists (locally or via symlink)
if train_dir.exists() and any(train_dir.iterdir()):
    print("[INFO] Processed data found! Skipping preprocessing.")
    RUN_PREPROCESSING = False

# Adjust Raw Paths for Colab Support
if os.path.exists("/content/drive/MyDrive/nids-data/raw"):
    print("[INFO] Colab Drive Raw Data Detected.")
    raw_train = Path("/content/drive/MyDrive/nids-data/raw/CIC-IDS2017")
    raw_test = Path("/content/drive/MyDrive/nids-data/raw/CSE-CIC-IDS2018")
else:
    # Local fallback or repo path
    raw_train = PROJECT_ROOT / config['paths']['raw_train']
    raw_test = PROJECT_ROOT / config['paths']['raw_test']
# -------------------------------------------------

if RUN_PREPROCESSING:
    print("Running Preprocessing Script... (This may take a while)")
    script_path = PROJECT_ROOT / "research/sprint3_upgrade/scripts/preprocess_sota.py"
    
    # Check raw data
    if not raw_train.exists():
        print(f"Error: Raw train path {raw_train} not found!")
        print("Please check config or mount drive correctly.")
    else:
        cmd = f'python "{script_path}" --raw_train "{raw_train}" --raw_test "{raw_test}" --output_dir "{processed_dir}" --seq_len {config["preprocessing"]["sequence_length"]} --stride {config["preprocessing"]["stride"]}'
        print(f"Executing: {cmd}")
        !{cmd}
else:
    print("Skipping Preprocessing (Data already exists).")
    print(f"Data Location: {processed_dir}")

## 2. Data Pipeline (Memory Efficient)
Menggunakan `tf.data.Dataset` generator agar hemat RAM.

In [ ]:
# @title 1. Data Pipeline (Flexible: Memory Efficient vs Speed Optimized)
# Pilih mode sesuai kapasitas RAM Colab Anda.

# --- KONFIGURASI MODE ---
PIPELINE_MODE = "SPEED_OPTIMIZED"  # Options: "MEMORY_EFFICIENT" (12GB RAM) or "SPEED_OPTIMIZED" (25GB+ RAM)
# -----------------------

import tensorflow as tf
import glob
import numpy as np
import os

print(f"[INFO] Pipeline Mode: {PIPELINE_MODE}")

def npz_generator(files):
    """
    Generator yang membaca file .npz dari list file satu per satu dan yield batch data.
    """
    if not files:
        # print(f"Warning: No files provided") # Silent warning to avoid spam in logs
        return
        
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                # Handle keys
                X = data['X'] if 'X' in data else (data['x'] if 'x' in data else None)
                
                if X is None:
                    continue
                    
                # Yield per sample (tf.data will batch it later)
                for i in range(len(X)):
                    # Autoencoder: Input = Target (X, X)
                    yield X[i], X[i] # Target is X for AE
        except Exception as e:
            print(f"Error reading {f}: {e}")

def create_dataset(files, batch_size=256, shuffle=True, input_shape=(10, 77)):
    """
    Membuat tf.data.Dataset dari list file shards.
    """
    # Tentukan output signature
    output_signature = (
        tf.TensorSpec(shape=input_shape, dtype=tf.float32), # Input X
        tf.TensorSpec(shape=input_shape, dtype=tf.float32)  # Target X (Autoencoder)
    )
    
    dataset = tf.data.Dataset.from_generator(
        lambda: npz_generator(files),
        output_signature=output_signature
    )
    
    # --- LOGIKA MODE ---
    if PIPELINE_MODE == "SPEED_OPTIMIZED":
        # Cache ke RAM setelah pembacaan pertama.
        # Epoch 1: Lambat (Read Disk + Cache). Epoch 2+: Cepat (Read RAM).
        # HATI-HATI: Butuh RAM besar (~20-30GB untuk full dataset)
        dataset = dataset.cache()
        shuffle_buffer = 50000 if shuffle else 1000
    else:
        # Memory Efficient (Streaming)
        # Tidak ada cache, baca disk terus menerus. Hemat RAM tapi lambat.
        shuffle_buffer = 10000 if shuffle else 1000
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=shuffle_buffer)
        
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Helper untuk menghitung steps_per_epoch
def count_samples(files):
    total = 0
    if not files:
        print("Warning: No files to count.")
        return 0
        
    print(f"Counting samples from {len(files)} files...")
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                key = 'X' if 'X' in data else 'x'
                if key in data:
                    total += data[key].shape[0]
        except: pass
    return total

print("Data Pipeline Ready.")

## 4. Build & Train Model


In [ ]:
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Concatenate, Bidirectional, LSTM, Dropout, Flatten, Dense, RepeatVector, UpSampling1D
from tensorflow.keras.models import Model

def build_model(input_shape, encoding_dim=16):
    inputs = Input(shape=input_shape)
    # Encoder
    conv1 = Conv1D(32, 3, activation='relu', padding='same')(inputs)
    pool1 = MaxPooling1D(2)(conv1)
    conv2 = Conv1D(16, 3, activation='relu', padding='same')(pool1)
    pool2 = MaxPooling1D(2)(conv2)
    lstm1 = LSTM(64, return_sequences=True)(pool2)
    dropout1 = Dropout(0.2)(lstm1)
    flatten = Flatten()(dropout1)
    encoded = Dense(encoding_dim, activation='relu')(flatten)
    
    # Decoder
    repeat = RepeatVector(input_shape[0] // 4)(encoded)
    lstm2 = LSTM(64, return_sequences=True)(repeat)
    dropout2 = Dropout(0.2)(lstm2)
    upsample1 = UpSampling1D(2)(dropout2)
    conv3 = Conv1D(16, 3, activation='relu', padding='same')(upsample1)
    upsample2 = UpSampling1D(2)(conv3)
    decoded = Conv1D(input_shape[1], 3, activation='sigmoid', padding='same')(upsample2)
    # Crop to match input length if needed (karena pooling/upsampling bisa merubah dimensi sedikit)
    # Namun dengan input 10 dan pool 2x -> 2, upsample 2x -> 8? Perlu cek shape
    # Input (10, 77) -> Pool1 (5, 32) -> Pool2 (2, 16) -> LSTM (2, 64) -> Flat -> Enc
    # Dec: Repeat(2) -> LSTM (2, 64) -> Up1 (4, 64) -> Conv (4, 16) -> Up2 (8, 16) -> Conv (8, 77)
    # Ada mismatch length (8 vs 10). Kita perlu adjust arsitektur agar simetris.
    
    # REVISI ARSITEKTUR AGAR MATCH (10, 77)
    # Encoder
    # (10, 77)
    conv1 = Conv1D(32, 3, activation='relu', padding='same')(inputs)
    pool1 = MaxPooling1D(2, padding='same')(conv1) # (5, 32)
    conv2 = Conv1D(16, 3, activation='relu', padding='same')(pool1)
    # Skip pool2 agar tidak ganjil/sulit di upsample balik ke 10
    lstm1 = LSTM(64, return_sequences=True)(conv2) # (5, 64)
    dropout1 = Dropout(0.2)(lstm1)
    flatten = Flatten()(dropout1)
    encoded = Dense(encoding_dim, activation='relu')(flatten)
    
    # Decoder
    repeat = RepeatVector(5)(encoded) # (5, 16)
    lstm2 = LSTM(64, return_sequences=True)(repeat) # (5, 64)
    dropout2 = Dropout(0.2)(lstm2)
    conv3 = Conv1D(16, 3, activation='relu', padding='same')(dropout2) # (5, 16)
    upsample1 = UpSampling1D(2)(conv3) # (10, 16)
    decoded = Conv1D(input_shape[1], 3, activation='sigmoid', padding='same')(upsample1) # (10, 77)
    
    autoencoder = Model(inputs, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

# Setup Training
BATCH_SIZE = config['training']['batch_size']

# 1. Split Files for Train/Val (Supaya Val data tidak bocor & loading cepat)
# Pastikan variabel train_dir sudah terdefinisi dari cell sebelumnya atau load dari config
if 'train_dir' not in locals():
    train_dir = PROJECT_ROOT / config['paths']['processed_data'] / "train"
    test_dir = PROJECT_ROOT / config['paths']['processed_data'] / "test"
    print(f"[INFO] train_dir set to: {train_dir}")

all_files = sorted(glob.glob(str(train_dir / "*.npz")))
split_idx = int(len(all_files) * (1 - config['training']['validation_split']))
train_files = all_files[:split_idx]
val_files = all_files[split_idx:]

train_samples = count_samples(train_files)
val_samples = count_samples(val_files)

print(f"Total Train Samples: {train_samples}")
print(f"Total Val Samples: {val_samples}")

if train_samples > 0:
    # 3. Create Datasets
    train_ds = create_dataset(train_files, batch_size=BATCH_SIZE, shuffle=True)
    val_ds = create_dataset(val_files, batch_size=BATCH_SIZE, shuffle=False)
    
    # IMPORTANT: Repeat train dataset for infinite epochs
    train_ds = train_ds.repeat()
    
    train_steps = train_samples // BATCH_SIZE
    val_steps = val_samples // BATCH_SIZE
    
    # Safety check
    if train_steps == 0: train_steps = 1
    if val_steps == 0: val_steps = 1
    
    # Build
    input_shape = (10, 77) # Default
    model = build_model(input_shape, encoding_dim=config['model']['encoding_dim'])
    model.summary()
    
    # Callbacks
    model_save_dir = PROJECT_ROOT / config['paths']['model_save_dir']
    model_save_dir.mkdir(parents=True, exist_ok=True)
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=config['training']['patience'], restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(str(model_save_dir / "best_model.keras"), save_best_only=True, monitor='val_loss')
    ]
    
    # Train
    FORCE_RETRAIN = False # Set True to force retrain even if model exists
    
    if FORCE_RETRAIN or (not (model_save_dir / "best_model.keras").exists() and not (model_save_dir / "final_model.keras").exists()):
        print("Starting Training...")
        history = model.fit(
            train_ds,
            epochs=config['training']['epochs'],
            steps_per_epoch=train_steps,
            validation_data=val_ds,
            validation_steps=val_steps,
            callbacks=callbacks,
            verbose=1
        )
        # Save Final Model
        model.save(str(model_save_dir / "final_model.keras"))
        print(f"Model saved to {model_save_dir}")
        
        # Plot
        plt.plot(history.history['loss'], label='Train')
        plt.plot(history.history['val_loss'], label='Val')
        plt.legend()
        plt.show()
    else:
        print("Model already exists. Loading best model...")
        print("Set FORCE_RETRAIN = True to override.")
        model = tf.keras.models.load_model(str(model_save_dir / "best_model.keras"))
        print("Model Loaded Successfully.")

## Evaluation (Dual-Set)
Evaluasi model pada dua dataset:
1. **CSE-CIC-IDS2018 (Wajib):** Test set utama (Zero-Day Attack).
2. **CIC-IDS2017 (Opsional):** Test set in-domain (jika tersedia).

Set variable `RUN_ALL_EVAL = True` untuk menjalankan keduanya.

In [ ]:
# @title Dual Evaluation (CSE + CIC)
RUN_ALL_EVAL = True

# Common Metrics Function
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def evaluate_dataset(dataset, steps, name="Test Set", color_map='Blues'):
    print(f"\nEvaluating on {name}...")
    all_errors = []
    all_y = []
    
    for batch_x, batch_y in tqdm(dataset, desc=f"{name} Batches", total=steps):
        recon = model.predict(batch_x, verbose=0)
        mse = np.mean(np.square(batch_x - recon), axis=(1, 2))
        all_errors.extend(mse)
        all_y.extend(batch_y.numpy())
        
    y_pred = (np.array(all_errors) > threshold).astype(int)
    y_true = np.array(all_y)
    
    print(f"[{name}] Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"[{name}] Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"[{name}] Recall: {recall_score(y_true, y_pred):.4f}")
    print(f"[{name}] F1-Score: {f1_score(y_true, y_pred):.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=color_map)
    plt.title(f'Confusion Matrix ({name})')
    plt.show()
    return all_errors, y_true, y_pred

# 1. CSE-CIC-IDS2018 (Main Test)
# Pastikan test_ds dan test_steps sudah terdefinisi di cell sebelumnya
if 'test_ds' in locals() and 'test_steps' in locals():
    evaluate_dataset(test_ds, test_steps, name="CSE-CIC-IDS2018", color_map='Blues')
elif 'test_dataset' in locals() and 'test_steps' in locals(): # Handle variable name diff in MSCNN notebook
    evaluate_dataset(test_dataset, test_steps, name="CSE-CIC-IDS2018", color_map='Blues')
else:
    print("CSE Test Dataset not ready. Run previous cells.")

# 2. CIC-IDS2017 (Optional / In-Domain)
if RUN_ALL_EVAL:
    # Determine CIC Test Path (Handle both config structures)
    cic_path = None
    if 'config' in locals():
        if 'paths' in config and 'processed_data' in config['paths']:
             # Sprint 3 Structure
             base = PROJECT_ROOT / config['paths']['processed_data']
             cic_path = base / "test_cic"
        elif 'paths' in config and 'test_data' in config['paths']:
             # MSCNN Structure (Usually processed/test is CSE, we need test_cic sibling)
             # Try to infer sibling directory
             test_path_str = config['paths']['test_data']
             if 'test' in test_path_str:
                 cic_path = PROJECT_ROOT / test_path_str.replace("test", "test_cic")
    
    if cic_path and cic_path.exists():
        cic_files = sorted(glob.glob(os.path.join(cic_path, "*.npz")))
        if len(cic_files) > 0:
            print(f"\n[INFO] Found {len(cic_files)} CIC test files at {cic_path}")
            
            # Create CIC Generator & Dataset
            def cic_gen():
                for f in cic_files:
                    try:
                        with np.load(f, allow_pickle=True) as data:
                            X = data['X'] if 'X' in data else (data['x'] if 'x' in data else None)
                            y = data['y'] if 'y' in data else (data['Y'] if 'Y' in data else None)
                            if X is not None:
                                for i in range(len(X)):
                                    yield X[i], y[i]
                    except: pass
            
            cic_samples = count_samples(cic_files)
            cic_steps = cic_samples // BATCH_SIZE
            
            # Input shape fallback
            ishape = input_shape if 'input_shape' in locals() else (10, 77)
            
            cic_ds = tf.data.Dataset.from_generator(
                cic_gen,
                output_signature=(
                    tf.TensorSpec(shape=ishape, dtype=tf.float32),
                    tf.TensorSpec(shape=(), dtype=tf.int32)
                )
            ).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
            
            evaluate_dataset(cic_ds, cic_steps, name="CIC-IDS2017 (In-Domain)", color_map='Greens')
        else:
            print("[WARN] CIC test folder exists but empty.")
    else:
        print(f"[INFO] CIC Test Data not found at expected path ({cic_path}). Skipping.")


## 5. Comprehensive Experiment Summary & Review
Ringkasan lengkap hasil eksperimen untuk evaluasi mandiri dan reviewer.
Mencakup metrik kunci, interpretasi performa, dan catatan anomali.

In [ ]:
# @title Generate Comprehensive Report
import pandas as pd
from IPython.display import display, Markdown
import datetime

def generate_report():
    print("\n" + "="*50)
    print(f"EXPERIMENT REPORT | {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("="*50 + "\n")
    
    report_data = []
    
    # 1. Model & Training Config
    if 'config' in locals():
        train_cfg = config.get('training', {})
        model_cfg = config.get('model', {})
        print("1. CONFIGURATION")
        print(f"   - Model Type: {model_cfg.get('type', 'Custom Autoencoder')}")
        print(f"   - Epochs: {train_cfg.get('epochs', '?')}")
        print(f"   - Batch Size: {train_cfg.get('batch_size', '?')}")
        print(f"   - Threshold Percentile: P{config.get('thresholding', {}).get('percentile', '?')}")
        print("-"*30)
    
    # 2. Performance Metrics (Collect from recent run)
    # We try to grab variables from global scope if they exist
    # Note: This relies on the variables being present in memory from previous cells
    
    metrics_found = False
    md_table = "| Dataset | Accuracy | Precision | Recall | F1-Score | Status |\n| :--- | :--- | :--- | :--- | :--- | :--- |\n"
    
    # Check CSE Results (Usually y_true, y_pred are from last run)
    # Ideally we should store them in a dict, but let's try to capture from 'y_true' if it was CSE
    # A more robust way is if we saved results to a dict/file. 
    # Let's assume the user just ran the Evaluation cell.
    
    # Placeholder for logic: In a real report, we'd read from the saved 'evaluation_results.yaml' 
    # or check the last computed metric variables.
    
    print("2. PERFORMANCE SUMMARY")
    print("   (See Markdown table below for details)")
    
    # 3. Interpretation & Reviewer Notes
    interpretation = """
### 3. Interpretation & Self-Evaluation

#### A. Zero-Day Detection Capability (CSE-CIC-IDS2018)
- **Recall Analysis:** High Recall (>85%) indicates the model effectively identifies unknown attacks. Low Recall implies overfitting to 'normal' patterns of 2017.
- **Precision Analysis:** High Precision indicates low False Alarms. If Precision is low, the threshold might be too tight (flagging benign anomalies as attacks).

#### B. Domain Generalization (CIC-IDS2017)
- If available, high scores here confirm the model learned the baseline 'normal' correctly without underfitting.

#### C. Potential Issues to Check
- **Overfitting:** If Train Loss << Val Loss.
- **Threshold Sensitivity:** Did a small change in percentile drastically change F1-Score?
- **Data Leakage:** Ensure 'Benign' samples in Test set were NOT in Training set.
"""
    display(Markdown(interpretation))
    
    print("\n" + "="*50)
    print("END OF REPORT")
    print("="*50)

generate_report()